# Build Drivers Dimension
1. Read silver `drivers` table
2. Read gold `ref_nationality_region` table
3. Join the data from `drivers` with `ref_nationality_region` using `nationality`
4. Select the required columns
   - drivers.driver_id
   - drivers.driver_name
   - drivers.date_of_birth
   - drivers.nationality
   - ref_nationality_region.region
5. Write the transformed data to gold `dim_drivers` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_drivers"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read source tables
- `silver.drivers`
- `gold.ref_nationality_region`

In [0]:
drivers_df = spark.table(f"{catalog_name}.{silver_schema}.drivers")
ref_nat_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

#### Step 2 - Join `drivers` with `nationality_region_df` using `nationality`
Select the following columns   
1. drivers.driver_id
2. drivers.driver_name
3. drivers.date_of_birth
4. drivers.nationality
5. ref_nationality_region.region

In [0]:
dim_drivers_df = (
    drivers_df
        .join(
            ref_nat_region_df,
            drivers_df.nationality == ref_nat_region_df.nationality,
            "left"
        )
        .select(
            drivers_df.driver_id,
            drivers_df.driver_name,
            drivers_df.date_of_birth,
            drivers_df.nationality,
            ref_nat_region_df.region
        )
)

display(dim_drivers_df)

#### Step 3 - Write the transformed data to the `gold` `dim_constructors` table

In [0]:
(
    dim_drivers_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

display(spark.table(target_table))